# Debug Think Tags in GRPO Training

This notebook troubleshoots why `<think>` tags are not being generated during GRPO training.

**Potential Causes:**
1. TRL version issue (need v0.25.0+ for proper `add_generation_prompt`)
2. Chat template not including `<think>` in generation prompt
3. Model type (Instruct vs Thinking variants)
4. `enable_thinking` parameter not being passed

In [ ]:
# Check versions
import trl
import transformers
print(f"TRL version: {trl.__version__}")
print(f"Transformers version: {transformers.__version__}")

# TRL v0.25.0+ has proper add_generation_prompt support
# If using older version, upgrade:
# !pip install trl>=0.25.0

## 1. CHECK IF THINK TAGS ARE SPECIAL TOKENS (Critical!)

This is likely the root cause - if `<think>` and `</think>` are special tokens, 
TRL's GRPOTrainer may be stripping them with `skip_special_tokens=True`.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load model in normal mode
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Koalacrown/qwen3-4b-cold-start-16bit",
    max_seq_length=4096,
    load_in_4bit=True,  # Use 4bit for debugging to save memory
    fast_inference=False,
)

# ============================================
# CRITICAL CHECK: Are think tags special tokens?
# ============================================
print("=" * 60)
print("CHECKING IF THINK TAGS ARE SPECIAL TOKENS")
print("=" * 60)

# Check all special tokens
print("\n1. All special tokens:")
print(f"   all_special_tokens: {tokenizer.all_special_tokens[:20]}...")  # First 20
print(f"   Total special tokens: {len(tokenizer.all_special_tokens)}")

# Check if <think> and </think> are in special tokens
think_open = "<think>"
think_close = "</think>"

print(f"\n2. Is '<think>' a special token? {think_open in tokenizer.all_special_tokens}")
print(f"   Is '</think>' a special token? {think_close in tokenizer.all_special_tokens}")

# Check added tokens (often where custom tokens go)
print(f"\n3. Added tokens vocab: {list(tokenizer.added_tokens_encoder.keys())[:20]}...")

# Encode think tags to see their token IDs
print("\n4. Token IDs for think tags:")
think_open_ids = tokenizer.encode(think_open, add_special_tokens=False)
think_close_ids = tokenizer.encode(think_close, add_special_tokens=False)
print(f"   '<think>' encodes to: {think_open_ids}")
print(f"   '</think>' encodes to: {think_close_ids}")

# Check if these IDs are in special_ids
special_ids = set(tokenizer.all_special_ids) if hasattr(tokenizer, 'all_special_ids') else set()
print(f"\n5. Are think token IDs special?")
for tid in think_open_ids:
    print(f"   Token ID {tid} is special: {tid in special_ids}")
for tid in think_close_ids:
    print(f"   Token ID {tid} is special: {tid in special_ids}")

# Test decode with and without skip_special_tokens
print("\n6. Decode comparison (skip_special_tokens):")
test_text = "<think>This is reasoning</think>Final answer"
test_ids = tokenizer.encode(test_text, add_special_tokens=False)
print(f"   Original: {test_text}")
print(f"   Token IDs: {test_ids}")
decoded_with_special = tokenizer.decode(test_ids, skip_special_tokens=False)
decoded_without_special = tokenizer.decode(test_ids, skip_special_tokens=True)
print(f"   Decoded (skip_special_tokens=False): {decoded_with_special}")
print(f"   Decoded (skip_special_tokens=True):  {decoded_without_special}")

# THE KEY TEST
print("\n" + "=" * 60)
if decoded_with_special != decoded_without_special:
    print("⚠️  THINK TAGS ARE BEING STRIPPED BY skip_special_tokens=True!")
    print("    This is likely why GRPO completions don't show think tags.")
    print("    TRL GRPOTrainer uses skip_special_tokens=True internally.")
else:
    print("✓ Think tags are NOT affected by skip_special_tokens")
print("=" * 60)

In [ ]:
# ============================================
# WORKAROUND: Patch tokenizer to not skip think tags
# ============================================
# If think tags are special tokens, we need to modify the tokenizer's behavior

# Option 1: Remove think tags from special tokens list (if they're there)
if think_open in tokenizer.all_special_tokens:
    print("Removing <think> from special tokens...")
    # This is tricky - special_tokens are usually read-only
    # We may need a different approach

# Option 2: Create a wrapper for decode that preserves think tags
class TokenizerWrapper:
    """Wrapper that intercepts decode calls and preserves think tags."""
    
    def __init__(self, tokenizer):
        self._tokenizer = tokenizer
        self._think_open_id = tokenizer.encode("<think>", add_special_tokens=False)
        self._think_close_id = tokenizer.encode("</think>", add_special_tokens=False)
        
    def __getattr__(self, name):
        return getattr(self._tokenizer, name)
    
    def decode(self, token_ids, skip_special_tokens=True, **kwargs):
        # Always preserve think tags by using skip_special_tokens=False
        # then manually remove other special tokens if needed
        result = self._tokenizer.decode(token_ids, skip_special_tokens=False, **kwargs)
        return result
    
    def batch_decode(self, sequences, skip_special_tokens=True, **kwargs):
        # Same logic for batch_decode
        return self._tokenizer.batch_decode(sequences, skip_special_tokens=False, **kwargs)

# Test the wrapper
print("\n7. Testing TokenizerWrapper:")
wrapped_tokenizer = TokenizerWrapper(tokenizer)
test_decoded = wrapped_tokenizer.decode(test_ids, skip_special_tokens=True)
print(f"   Wrapped decode result: {test_decoded}")
print(f"   Contains think tags: {'<think>' in test_decoded and '</think>' in test_decoded}")

In [ ]:
# ============================================
# RAW MODEL OUTPUT TEST - See what tokens are generated
# ============================================
print("=" * 60)
print("RAW MODEL OUTPUT (Token-level inspection)")
print("=" * 60)

# Prepare model for inference
FastLanguageModel.for_inference(model)

# Create a prompt that should trigger thinking
test_messages = [
    {"role": "system", "content": "Think step by step in <think></think> tags before answering."},
    {"role": "user", "content": "What is 2+2?"}
]

# Apply chat template
prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
print(f"\nPrompt ends with:\n{repr(prompt[-150:])}")

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
print(f"\nInput token count: {inputs.input_ids.shape[1]}")

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
)

# Get only the new tokens
new_token_ids = outputs[0][inputs.input_ids.shape[1]:]
print(f"Generated token count: {len(new_token_ids)}")

# Show raw token IDs
print(f"\nFirst 30 generated token IDs:")
print(new_token_ids[:30].tolist())

# Decode token by token to see exactly what's there
print(f"\nToken-by-token decode (first 30 tokens):")
for i, tid in enumerate(new_token_ids[:30].tolist()):
    token_str = tokenizer.decode([tid], skip_special_tokens=False)
    is_special = tid in (tokenizer.all_special_ids if hasattr(tokenizer, 'all_special_ids') else [])
    print(f"  [{i:2d}] ID={tid:6d} | special={is_special} | '{token_str}'")

# Full decode with and without skip_special_tokens
print(f"\n" + "-" * 40)
print("Full decode comparison:")
full_with = tokenizer.decode(new_token_ids, skip_special_tokens=False)
full_without = tokenizer.decode(new_token_ids, skip_special_tokens=True)

print(f"\nskip_special_tokens=False ({len(full_with)} chars):")
print(full_with[:300])

print(f"\nskip_special_tokens=True ({len(full_without)} chars):")
print(full_without[:300])

# Check for think tags in both
print(f"\n" + "=" * 60)
print(f"<think> in skip_special_tokens=False: {'<think>' in full_with}")
print(f"<think> in skip_special_tokens=True:  {'<think>' in full_without}")
print(f"</think> in skip_special_tokens=False: {'</think>' in full_with}")
print(f"</think> in skip_special_tokens=True:  {'</think>' in full_without}")
print("=" * 60)

In [ ]:
# ============================================
# Test 3: Check what GRPOTrainer does internally
# ============================================
print("=" * 60)
print("SIMULATING GRPO TRAINER BEHAVIOR")
print("=" * 60)

# GRPOTrainer typically:
# 1. Applies chat template with add_generation_prompt=True
# 2. Generates completions
# 3. Decodes with skip_special_tokens (check TRL source)
# 4. Passes decoded text to reward functions

# Let's simulate this:
from unsloth.chat_templates import get_chat_template

# Apply unsloth's qwen3-thinking template
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-thinking")

test_messages = [{"role": "user", "content": "What is 2+2?"}]

# Step 1: Apply template
prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
print(f"1. Chat template applied. Prompt ends with:")
print(repr(prompt[-100:]))

# Step 2: Generate (simulated with known output)
# In real GRPO, model.generate() is called
simulated_completion_ids = tokenizer.encode(
    "<think>Let me calculate: 2+2=4</think>The answer is 4",
    add_special_tokens=False
)
print(f"\n2. Simulated completion token IDs: {simulated_completion_ids[:10]}...")

# Step 3: Decode (this is where the issue might be)
# Check what TRL does - it likely uses batch_decode with skip_special_tokens=True
decoded_true = tokenizer.decode(simulated_completion_ids, skip_special_tokens=True)
decoded_false = tokenizer.decode(simulated_completion_ids, skip_special_tokens=False)

print(f"\n3. Decoded with skip_special_tokens=True:")
print(f"   '{decoded_true}'")
print(f"   Has <think>: {'<think>' in decoded_true}")

print(f"\n   Decoded with skip_special_tokens=False:")
print(f"   '{decoded_false}'")
print(f"   Has <think>: {'<think>' in decoded_false}")

# Step 4: This is what reward function sees
print(f"\n4. What reward function receives:")
print(f"   If TRL uses skip_special_tokens=True, think tags are {'MISSING' if '<think>' not in decoded_true else 'PRESENT'}")

print("\n" + "=" * 60)
if decoded_true != decoded_false:
    print("⚠️  CONFIRMED: skip_special_tokens=True removes think tags!")
    print("   This is why your GRPO reward function sees no think tags.")
    print("\n   SOLUTION: Upgrade TRL or patch the tokenizer (see workaround above)")
else:
    print("✓ Think tags are preserved with both settings")
print("=" * 60)

## 2. Apply Unsloth Chat Template

In [ ]:
from unsloth.chat_templates import get_chat_template

# Apply qwen3-thinking template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-thinking",
)

print("After applying qwen3-thinking template:")
result = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
print(repr(result[-150:]))
print(f"\n<think> in output: {'<think>' in result}")

In [ ]:
# Check updated template
print("=" * 60)
print("UPDATED CHAT TEMPLATE")
print("=" * 60)
print(tokenizer.chat_template)

## 3. Manual Template Fix

If `<think>` is not in the template, we need to manually add it.

In [ ]:
# Check if template has add_generation_prompt section
original_template = tokenizer.chat_template

if "add_generation_prompt" in original_template:
    print("Found add_generation_prompt in template")
    
    # Find what's in the add_generation_prompt section
    idx = original_template.find("add_generation_prompt")
    print(f"\nSection around add_generation_prompt:")
    print(repr(original_template[idx:idx+200]))
else:
    print("WARNING: add_generation_prompt NOT found in template!")

In [ ]:
# Fix: Manually patch the template to include <think>
def patch_template_for_think(tokenizer):
    """Patch the chat template to include <think> in generation prompt."""
    original = tokenizer.chat_template
    
    # Check if already has <think> in add_generation_prompt
    if "add_generation_prompt" in original and "<think>" not in original:
        # Pattern 1: {% if add_generation_prompt %}{{ '<|assistant|>' }}{% endif %}
        # We need to add <think> after the assistant tag
        
        # Try different patterns
        patterns_to_try = [
            ("{% if add_generation_prompt %}", "{% if add_generation_prompt %}<think>\n"),
            ("<|im_start|>assistant\n{% endif %}", "<|im_start|>assistant\n<think>\n{% endif %}"),
            ("'<|im_start|>assistant\n' }}{% endif %}", "'<|im_start|>assistant\n<think>\n' }}{% endif %}"),
        ]
        
        patched = original
        for old, new in patterns_to_try:
            if old in patched:
                patched = patched.replace(old, new)
                print(f"Applied patch: {old[:50]}... -> {new[:50]}...")
                break
        
        if patched != original:
            tokenizer.chat_template = patched
            print("Template patched successfully!")
            return True
        else:
            print("Could not find pattern to patch. Manual intervention needed.")
            return False
    elif "<think>" in original:
        print("Template already contains <think>")
        return True
    else:
        print("Template structure not recognized")
        return False

# Apply patch
patch_template_for_think(tokenizer)

In [ ]:
# Test after patching
print("=" * 60)
print("AFTER PATCHING")
print("=" * 60)

result = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
print("Result:")
print(repr(result[-150:]))
print(f"\n<think> in output: {'<think>' in result}")

## 4. Test Model Generation

In [ ]:
# Prepare model for inference
FastLanguageModel.for_inference(model)

# Test generation
test_prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("Input prompt (last 200 chars):")
print(repr(test_prompt[-200:]))
print("\n" + "=" * 60)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)

generated = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
print("\nGenerated output:")
print(generated[:500])

print(f"\n<think> in output: {'<think>' in generated}")
print(f"</think> in output: {'</think>' in generated}")

## 5. Simple GRPO Test with Think Tags

In [ ]:
# Reload model with LoRA for training
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",  # Try base Qwen3 to rule out cold-start model issues
    max_seq_length=2048,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# Set up custom chat template that FORCES <think> tag
CUSTOM_TEMPLATE = """{% for message in messages %}
{% if message['role'] == 'system' %}
<|im_start|>system
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'user' %}
<|im_start|>user
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'assistant' %}
<|im_start|>assistant
{{ message['content'] }}<|im_end|>
{% endif %}
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
<think>
{% endif %}"""

tokenizer.chat_template = CUSTOM_TEMPLATE

# Test it
test_msgs = [{"role": "user", "content": "What is 2+2?"}]
result = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
print("Custom template result:")
print(result)
print(f"\nEnds with <think>: {result.strip().endswith('<think>')}")

In [ ]:
# Create minimal dataset
from datasets import Dataset

SYSTEM_PROMPT = """Think step by step. Put your reasoning in <think></think> tags, then give your answer."""

data = [
    {"prompt": [{"role": "system", "content": SYSTEM_PROMPT}, 
                {"role": "user", "content": "What is 15 + 27?"}]},
    {"prompt": [{"role": "system", "content": SYSTEM_PROMPT}, 
                {"role": "user", "content": "What is the capital of France?"}]},
    {"prompt": [{"role": "system", "content": SYSTEM_PROMPT}, 
                {"role": "user", "content": "Explain why the sky is blue in one sentence."}]},
]

dataset = Dataset.from_list(data)
print(f"Dataset: {len(dataset)} examples")

In [ ]:
# Simple reward function that checks for think tags
import re

def reward_think_tags(completions, **kwargs):
    """Reward for proper <think>...</think> format."""
    scores = []
    for completion in completions:
        content = completion[0]["content"]
        
        # Debug: print first completion
        if len(scores) == 0:
            print(f"\n[DEBUG] First completion ({len(content)} chars):")
            print(content[:300])
            print(f"Has <think>: {'<think>' in content}")
            print(f"Has </think>: {'</think>' in content}")
        
        # Check for think tags
        has_think = "<think>" in content and "</think>" in content
        
        if has_think:
            # Extract think content
            match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
            if match and len(match.group(1)) > 20:
                scores.append(1.0)  # Good thinking
            else:
                scores.append(0.5)  # Has tags but short
        else:
            scores.append(-1.0)  # No tags
    
    return scores

print("Reward function defined")

In [ ]:
# Configure GRPO
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./debug_grpo",
    
    # Generation settings
    temperature=0.8,
    max_completion_length=256,
    max_prompt_length=512,
    num_generations=4,
    
    # Training settings
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_steps=5,  # Just a few steps for debugging
    
    # Logging
    logging_steps=1,
    report_to="none",
)

print("GRPOConfig ready")

In [ ]:
# Create trainer and run
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_think_tags],
    args=training_args,
    train_dataset=dataset,
)

print("Starting training (5 steps)...")
trainer.train()

## 6. Alternative: Use Qwen3-Thinking Model

If the regular Qwen3 model doesn't produce think tags, try the explicit thinking variant.

In [ ]:
# Try loading the explicit thinking model
model_thinking, tokenizer_thinking = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Thinking-2507",  # Explicit thinking model
    max_seq_length=2048,
    load_in_4bit=True,
    fast_inference=False,
)

# Test generation
test_msgs = [{"role": "user", "content": "What is 2+2?"}]
prompt = tokenizer_thinking.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
print("Prompt ends with:")
print(repr(prompt[-100:]))

## Summary: Common Fixes

### Issue 1: Think tags are special tokens being stripped

If `skip_special_tokens=True` removes think tags, you have these options:

1. **Upgrade TRL** to get the fix: `pip install trl>=0.25.0`

2. **Patch GRPOTrainer** to use `skip_special_tokens=False`:
```python
# Monkey-patch before creating trainer
import trl.trainer.grpo_trainer as grpo_module

original_batch_decode = grpo_module.GRPOTrainer._batch_decode
def patched_batch_decode(self, *args, **kwargs):
    kwargs['skip_special_tokens'] = False
    return original_batch_decode(self, *args, **kwargs)
grpo_module.GRPOTrainer._batch_decode = patched_batch_decode
```

3. **Use TokenizerWrapper** that overrides decode behavior

### Issue 2: Chat template not including `<think>` in generation prompt

Use the custom template that forces `<think>`:
```python
tokenizer.chat_template = '''{% for message in messages %}...
{% if add_generation_prompt %}<|im_start|>assistant
<think>
{% endif %}'''
```

### Issue 3: Model wasn't trained with think tags

If using an **Instruct** model instead of **Thinking** model:
- Switch to `unsloth/Qwen3-4B-Thinking-2507`
- Or fine-tune the model to use think tags first (SFT before GRPO)